In [64]:
import requests
import pandas as pd

pd.set_option("display.max_columns", None)

In [65]:
API_KEY = "jobboerse-jobsuche"
BASE_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/app/jobs"

headers = {
    "X-API-Key": API_KEY
}

### Erste Datensammlung

Es werden mehrere Suchbegriffe verwendet, um verschiedene Berufsfelder abzudecken. Pro Suchbegriff werden mehrere Seiten geladen. Ziel ist noch nicht ein perfekter Enddatensatz, sondern ein erster Rohdatensatz für die Datenexploration.

In [66]:
search_terms = ["data", "marketing", "logistik", "verkauf", "software", "pflege"]

all_jobs = []

for term in search_terms:
    print("Lade Daten für:", term)

    for page in range(1, 4):
        params = {
            "was": term,
            "page": page,
            "size": 100
        }

        response = requests.get(BASE_URL, headers=headers, params=params)

        if response.status_code != 200:
            print("Fehler bei", term, "Seite", page, "Status:", response.status_code)
            continue

        data = response.json()
        jobs = data.get("stellenangebote", [])

        print("  Seite", page, "-", len(jobs), "Anzeigen")

        for job in jobs:
            all_jobs.append(job)

print("\nGesamtzahl geladener Rohdatensätze:", len(all_jobs))

Lade Daten für: data
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen
Lade Daten für: marketing
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen
Lade Daten für: logistik
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen
Lade Daten für: verkauf
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen
Lade Daten für: software
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen
Lade Daten für: pflege
  Seite 1 - 100 Anzeigen
  Seite 2 - 100 Anzeigen
  Seite 3 - 100 Anzeigen

Gesamtzahl geladener Rohdatensätze: 1800


In [67]:
if len(all_jobs) > 0:
    print("Beispielhafte Keys eines Job-Objekts:\n")
    print(all_jobs[0].keys())

Beispielhafte Keys eines Job-Objekts:

dict_keys(['beruf', 'titel', 'refnr', 'arbeitsort', 'arbeitgeber', 'aktuelleVeroeffentlichungsdatum', 'modifikationsTimestamp', 'eintrittsdatum', 'kundennummerHash'])


In [68]:
def get_nested_value(d, keys, default=None):
    value = d
    for key in keys:
        if isinstance(value, dict) and key in value:
            value = value[key]
        else:
            return default
    return value


rows = []

for job in all_jobs:
    row = {
        "refnr": job.get("refnr"),
        "titel": job.get("titel"),
        "beruf": job.get("beruf"),
        "arbeitgeber": job.get("arbeitgeber"),
        "eintrittsdatum": job.get("eintrittsdatum"),
        "aktuelleVeroeffentlichungsdatum": job.get("aktuelleVeroeffentlichungsdatum"),
        "ort": get_nested_value(job, ["arbeitsort", "ort"]),
        "region": get_nested_value(job, ["arbeitsort", "region"]),
        "plz": get_nested_value(job, ["arbeitsort", "plz"]),
        "land": get_nested_value(job, ["arbeitsort", "land"]),
    }
    rows.append(row)

df = pd.DataFrame(rows)

print("Form des DataFrames:", df.shape)
df.head()

Form des DataFrames: (1800, 10)


,refnr,titel,beruf,arbeitgeber,eintrittsdatum,aktuelleVeroeffentlichungsdatum,ort,region,plz,land
0,12265-500563_JB5112036-S,Data Engineer (m/w/d),Data Engineer,FERCHAU GmbH Niederlassung Bielefeld,2026-04-01,2026-04-01,Bielefeld,Nordrhein-Westfalen,33602,Deutschland
1,12265-500191_JB5110630-S,Data Engineer (m/w/d),Data Engineer,FERCHAU GmbH Niederlassung Rosenheim,2026-03-31,2026-03-31,"Engelsberg, Oberbayern",Bayern,84549,Deutschland
2,12117-28699084-YF-S,Data Engineer (m/w/d),Data Engineer,BTB Blockheizkraftwerks- Träger- und Betreiber...,2026-04-01,2026-04-01,Berlin,Berlin,10585,Deutschland
3,14225-1d22a8038537e450-S,Data Engineer (m/w/d),Data Engineer,Riverty Group GmbH,2026-04-01,2026-04-01,Berlin,Berlin,10623,Deutschland
4,11949-17172996-S,Data Engineer,Data Engineer,PwC Transaction Services Wirtschaftsprüfung GmbH,2026-04-01,2026-03-31,"Wien,Donaustadt",Wien,1220,Österreich
